In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [9]:
train_data = datasets.ImageFolder(r'D:\machineLearning\dog_breed\dataset_split\train', transform=train_tf)
val_data   = datasets.ImageFolder(r'D:\machineLearning\dog_breed\dataset_split\val',   transform=val_tf)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=16)

In [10]:
CLASS_NAMES = train_data.classes  # Save breed names
print("Breeds:", CLASS_NAMES)

Breeds: ['Beagle', 'Boxer', 'Bulldog', 'Dachshund', 'German_Shepherd', 'Golden_Retriever', 'Labrador_Retriever', 'Poodle', 'Rottweiler', 'Yorkshire_Terrier']


In [11]:
model = models.efficientnet_b0(pretrained=True)
for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(1280, len(CLASS_NAMES))
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

c:\Users\athav\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\athav\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
best_val_loss = float('inf')

In [13]:
for epoch in range(25):
    # Train
    model.train()
    train_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            val_loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss/len(train_loader):.3f} "
          f"| Val Loss: {val_loss/len(val_loader):.3f} | Val Acc: {acc:.1f}%")

    scheduler.step(val_loss)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state': model.state_dict(),
            'class_names': CLASS_NAMES
        }, '../model.pth')
        print("  ✅ Best model saved!")

print("Training complete!")

Epoch 01 | Train Loss: 1.970 | Val Loss: 1.583 | Val Acc: 75.0%
  ✅ Best model saved!
Epoch 02 | Train Loss: 1.396 | Val Loss: 1.144 | Val Acc: 86.5%
  ✅ Best model saved!
Epoch 03 | Train Loss: 1.115 | Val Loss: 0.946 | Val Acc: 84.9%
  ✅ Best model saved!
Epoch 04 | Train Loss: 0.946 | Val Loss: 0.817 | Val Acc: 87.5%
  ✅ Best model saved!
Epoch 05 | Train Loss: 0.851 | Val Loss: 0.711 | Val Acc: 90.6%
  ✅ Best model saved!
Epoch 06 | Train Loss: 0.733 | Val Loss: 0.611 | Val Acc: 91.1%
  ✅ Best model saved!
Epoch 07 | Train Loss: 0.691 | Val Loss: 0.563 | Val Acc: 90.6%
  ✅ Best model saved!
Epoch 08 | Train Loss: 0.641 | Val Loss: 0.496 | Val Acc: 91.1%
  ✅ Best model saved!
Epoch 09 | Train Loss: 0.591 | Val Loss: 0.522 | Val Acc: 90.6%
Epoch 10 | Train Loss: 0.522 | Val Loss: 0.456 | Val Acc: 91.7%
  ✅ Best model saved!
Epoch 11 | Train Loss: 0.513 | Val Loss: 0.451 | Val Acc: 91.1%
  ✅ Best model saved!
Epoch 12 | Train Loss: 0.479 | Val Loss: 0.398 | Val Acc: 91.7%
  ✅ Best mod